In [0]:
dbutils.library.restartPython()

In [0]:
# Cell 2: Enable AI Gateway inference logging
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# AI Gateway inference tables are automatically enabled for model serving endpoints
# No explicit configuration needed - they're created automatically!

endpoint = w.serving_endpoints.get("gdpr-agent-staging")
print(f"✅ Endpoint: {endpoint.name}")
print(f"   State: {endpoint.state.ready if endpoint.state else 'UNKNOWN'}")
print(f"   URL: {endpoint.url}")

print("\n📊 Inference logs will be available in:")
print("   System table: system.serving.endpoint_request_logs")
print("\n   Query your logs with:")
print(f"   SELECT * FROM system.serving.endpoint_request_logs")
print(f"   WHERE endpoint_name = 'gdpr-agent-staging'")
print(f"   AND date >= current_date() - 7")

In [0]:
# Cell 3: Check if inference logs exist
query = """
SELECT 
    date,
    endpoint_name,
    request_id,
    status_code,
    execution_duration_ms,
    COUNT(*) as request_count
FROM system.serving.endpoint_request_logs
WHERE endpoint_name = 'gdpr-agent-staging'
  AND date >= current_date() - 7
GROUP BY date, endpoint_name, request_id, status_code, execution_duration_ms
LIMIT 10
"""

try:
    df = spark.sql(query)
    if df.count() > 0:
        print("✅ Inference logs found!")
        display(df)
    else:
        print("⚠️  No logs yet. Make a request to the endpoint first.")
except Exception as e:
    print(f"❌ Error: {e}")
    print("\nMake sure you have access to system.serving tables.")

In [0]:
# Simple imports
from monitoring import run_monitoring, config

# Direct monitor imports
from monitoring.monitors import QualityMonitor, PerformanceMonitor

# Utility imports
from monitoring.utils import DatabricksClient, LLMJudge

# Reports
from monitoring.reports import prepare_dashboard_data

# Run monitoring
results = run_monitoring(days_back=7, sample_size=20)

# Or prepare dashboard data
prepare_dashboard_data(days_back=30)